# AbovePy Advanced — Multi-County Mosaic + Occulus Pipeline

This notebook demonstrates advanced AbovePy + Occulus workflows:

1. Search and download multiple tiles across a county boundary
2. Mosaic tiles (AbovePy) into a single raster
3. Load mosaicked or tiled LiDAR into Occulus
4. Run a full forest/terrain inventory across a watershed
5. Export results as CSV and GeoTIFF-ready arrays

## Setup
```bash
pip install abovepy[lidar]
pip install occulus[all]
```

In [ ]:
try:
    import abovepy
    print(f'AbovePy {abovepy.__version__}')
    ABOVE_AVAILABLE = True
except ImportError:
    print('AbovePy not installed: pip install abovepy[lidar]')
    ABOVE_AVAILABLE = False

import occulus
print(f'Occulus  {occulus.__version__}')

## 1. Search by Bounding Box

Search for all available LiDAR tiles within a geographic bounding box —
e.g. the Red River Gorge area spanning Powell and Wolfe counties, KY.

In [ ]:
if ABOVE_AVAILABLE:
    from abovepy import search
    
    # Red River Gorge watershed — straddles Powell/Wolfe/Menifee counties
    bbox_rrg = [-83.72, 37.72, -83.50, 37.85]  # [W, S, E, N] WGS84
    
    tiles = search('lidar', bbox=bbox_rrg, phase=3)
    print(f'Red River Gorge — {len(tiles)} Phase 3 tiles found')
    for t in tiles[:5]:
        print(f'  {t}')

## 2. Download Multiple Tiles

In [ ]:
if ABOVE_AVAILABLE and tiles:
    from abovepy import download
    import tempfile
    from pathlib import Path
    
    dl_dir = Path(tempfile.mkdtemp()) / 'rrg_lidar'
    dl_dir.mkdir(parents=True, exist_ok=True)
    
    # Download first 4 tiles (limit for demo)
    paths = []
    for tile in tiles[:4]:
        p = download(tile, dest=dl_dir)
        paths.append(p)
        print(f'  ✓ {p.name} ({p.stat().st_size/1e6:.1f} MB)')

## 3. Process Each Tile and Aggregate Results

In [ ]:
if ABOVE_AVAILABLE and 'paths' in dir() and paths:
    import numpy as np
    from occulus.io import read
    from occulus.metrics import compute_cloud_statistics, canopy_height_model, coverage_statistics
    from occulus.segmentation import classify_ground_csf, segment_trees
    
    results = []
    for p in paths:
        print(f'Processing {p.name}…')
        cloud = read(p, platform='aerial', subsample=0.2)
        classified = classify_ground_csf(cloud)
        
        try:
            chm, _, _ = canopy_height_model(classified, resolution=1.0)
            cov = coverage_statistics(classified, resolution=1.0)
            seg = segment_trees(classified, resolution=1.0, min_height=3.0)
            stats = compute_cloud_statistics(cloud)
            results.append({
                'tile': p.name,
                'n_points': cloud.n_points,
                'max_height': float(np.nanmax(chm)),
                'gap_fraction': cov.gap_fraction,
                'trees': seg.n_segments,
                'z_range': stats.z_max - stats.z_min,
            })
        except Exception as e:
            print(f'  Skipped: {e}')
    
    print('\n=== Watershed Summary ===')
    for r in results:
        print(f"  {r['tile'][:30]:30s} | trees={r['trees']:4d} | max ht={r['max_height']:.1f} m | gap={r['gap_fraction']:.2%}")

## 4. Mosaic DEM (AbovePy)

In [ ]:
if ABOVE_AVAILABLE:
    try:
        from abovepy import mosaic
        import tempfile
        from pathlib import Path
        
        # Mosaic DEM tiles for the bounding box
        out_tif = Path(tempfile.mkdtemp()) / 'rrg_dem.tif'
        mosaic('dem', bbox=bbox_rrg, phase=2, dest=out_tif)
        print(f'DEM mosaic saved: {out_tif} ({out_tif.stat().st_size/1e6:.1f} MB)')
    except Exception as e:
        print(f'Mosaic: {e}')

## 5. Stream Point Data Without Downloading

In [ ]:
if ABOVE_AVAILABLE:
    from abovepy import read as above_read
    import numpy as np
    from occulus.types import AerialCloud
    
    # Small window: Natural Bridge State Resort Park
    bbox_nb = [-83.688, 37.773, -83.670, 37.786]
    
    try:
        df = above_read('lidar', bbox=bbox_nb, phase=3)
        print(f'Streamed {len(df):,} points (no download)')
        
        cloud = AerialCloud(df[['x', 'y', 'z']].values.astype(np.float64))
        from occulus.metrics import compute_cloud_statistics
        stats = compute_cloud_statistics(cloud)
        print(f'Natural Bridge — Z: {stats.z_min:.0f}–{stats.z_max:.0f} m  ({stats.z_max-stats.z_min:.0f} m relief)')
    except Exception as e:
        print(f'Stream read: {e}')

---

**AbovePy resources**:
- PyPI: https://pypi.org/project/abovepy/
- KY From Above portal: https://kyfromabove.ky.gov/
- Data products: LiDAR (Phases 1–3), DEM (1ft/2ft), Orthoimagery (6in/3in)

**Next**: `12_complete_workflow.ipynb`